# Limpieza de los Datos de la Tabla bronce.clientes para Cargalos en la Capa Plata

Proposito del script:  
- Verificar columna por columna los tipos de datos para encontrar inconsistencias en los datos.  
- Limpiar y estandarizar, columna por columna los datos.  
- Crear un archivo para los registros con inconsistencias con nombre "revision_clientes.parquet".
- Exportar la nueva tabla como un archivo con nombre "semi_limpio_clientes.parquet".

# Estableciendo la Conexion

In [1]:
# Importando Librerias y Cargando Archivos
import pandas as pd 
from funciones import limpiar_texto,formato_genero,formato_nivel_educacion
from conexiones_y_rutas import obtener_engine,obtener_ruta_archivo
from datetime import date

engine = obtener_engine()

df_clientes = pd.read_sql(
    "SELECT * FROM bronce.clientes",
    con=engine
)
df_clientes_tra = df_clientes.copy()

# Archivos de Ayuda

**Nota**: Este archivo se va utilizar para verificar las siguientes columnas:  
- [Ciudad Residencia](#ciudad_residencia)
- [Departamento Residencia](#departamento_residencia)
- [Region Residencia](#region_residencia)

In [2]:
# df del registro de ciudades de la empresa
df_ciudades = pd.read_csv(obtener_ruta_archivo("archivos_de_ayuda","peru_ciudades.csv"))
df_ciudades_tra = df_ciudades.copy()
df_ciudades_tra.head()

,ciudad,departamento,region
0,Tumbes,Tumbes,Norte
1,Zarumilla,Tumbes,Norte
2,Zorritos,Tumbes,Norte
3,Piura,Piura,Norte
4,Sullana,Piura,Norte


In [3]:
# se va a usar para validar sucursal_id y fecha_registro 
df_sucursales = pd.read_parquet(obtener_ruta_archivo("archivos_semi_limpios","semi_limpio_sucursales.parquet"))
df_sucursales_tra = df_sucursales.copy()
df_sucursales_tra.head()

,sucursal_id,codigo_sucursal,nombre_sucursal,tipo_sucursal,ciudad,departamento,region,zona,fecha_apertura,estado_sucursal
0,1,SUC0001,Oficina Principal Lima,Oficina Principal,San Juan de Lurigancho,Lima,Lima y Callao,Urbano,2005-07-24,Activa
1,2,SUC0002,Agencia Santiago de Surco 1,Agencia,Santiago de Surco,Lima,Lima y Callao,Urbano,2010-01-03,Activa
2,3,SUC0003,Agencia Miraflores 2,Punto de Atención,Miraflores,Lima,Lima y Callao,Urbano,2007-04-20,Activa
3,4,SUC0004,Agencia los Olivos 3,Agencia,Los Olivos,Lima,Lima y Callao,Urbano,2014-06-19,Activa
4,5,SUC0005,Agencia Lima 4,Agencia,Lima,Lima,Lima y Callao,Urbano,2007-02-07,Activa


# Resumen de las Columnas

- **cliente_id**: Identificador unico de cada cliente.  
- **tipo_documento**: Documento de registro del cliente (ejem: DNI o CE).  
- **numero_documento**: Numero del documento del cliente, la cantidad de digitos depende del tipo de documento.  
- **nombres**: Nombres del clientes.  
- **apellido_paterno**: Apellido paterno del cliente.  
- **apellido_materno**: Apellido materno del cliente.  
- **fecha_nacimiento**: Fecha de nacimiento del cliente.  
- **edad**: Edad actual del cliente.   
- **genero**: Genero del cliente (ejem: Masculino o Femenino).  
- **estado_civil**: Estado civil del cliente (ejem: Soltero, Casado).  
- **nivel_educacion**: Nivel de educacion del cliente (ejem: Secundaria o Universitaria).  
- **ocupacion**: Ocupacion actual del cliente (ejem: Empleado Publico y Empresario).  
- **sector_economico**: Sector economico donde se desarrolla el cliente (ejem: Finanzas o Transporte).  
- **ciudad_residencia**: Ciudad de residencia del cliente (ejmp: Santiago de Surco o La Esperanza).  
- **departamento_residencia**: Departamento de residencia del cliente (ejmp: Ica o Arequipa).  
- **region_residencia**: Region de residencia del cliente (ejmp: Norte o Sur).  
- **ingresos_mensuales**: Ingresos mensuales del cliente.  
- **egresos_mensuales**: Egresos mensuales del cliente.  
- **patrimonio_estimado**: Valuacion del patrimonio estimado del cliente.  
- **score_crediticio**: Score crediticio del cliente (0-1000).  
- **segmento_cliente**: Segmento del cliente (ejem: Regular, Premium o VIP).  
- **canal_captacion**: Canal de captacion del cliente (ejem: Agencia o Digital).  
- **fecha_registro**: Fecha de registro de cliente en el banco.  
- **antiguedad_clientes_meses**: Antiguedad de cliente en meses.  
- **sucursal_id**: Identificador unico de la sucursal donde esta inscrito cada cliente.  
- **estado_cliente**: Estado actual del cliente (ejem: Activo, Inactivo o Bloqueado).  

# Verificacion de la Calidad y Limpieza de los Datos

In [4]:
df_clientes.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5005 entries, 0 to 5004
Data columns (total 26 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   cliente_id                5005 non-null   int64  
 1   tipo_documento            5005 non-null   object 
 2   numero_documento          5005 non-null   object 
 3   nombres                   5005 non-null   object 
 4   apellido_paterno          5005 non-null   object 
 5   apellido_materno          5005 non-null   object 
 6   fecha_nacimiento          5005 non-null   object 
 7   edad                      5005 non-null   int64  
 8   genero                    5005 non-null   object 
 9   estado_civil              5005 non-null   object 
 10  nivel_educacion           4921 non-null   object 
 11  ocupacion                 4892 non-null   object 
 12  sector_economico          5005 non-null   object 
 13  ciudad_residencia         5005 non-null   object 
 14  departam

In [5]:
df_clientes_tra.head()

,cliente_id,tipo_documento,numero_documento,nombres,apellido_paterno,apellido_materno,fecha_nacimiento,edad,genero,estado_civil,...,ingresos_mensuales,egresos_mensuales,patrimonio_estimado,score_crediticio,segmento_cliente,canal_captacion,antiguedad_cliente_meses,fecha_registro,sucursal_id,estado_cliente
0,1,DNI,60366909,Carmen,Ramírez,Flores,16/09/1977,47,F,Casado,...,-2072.19,888.76,39631.77,550,Regular,Agencia,105,2016/05/07,14,Activo
1,2,DNI,62729806,Cecilia,Cusi,Cusi,13/03/1964,60,f,Casado,...,5663.68,3037.05,125691.99,528,Regular,Digital,87,2017-10-18,13,Activo
2,3,CE,641708053,Juan,Ortiz,Medina,1964-08-01,60,M,Casado,...,4750.66,2302.13,226055.14,493,Regular,Agencia,72,2019/01/08,13,Activo
3,4,DNI,29912419,Carlos,Flores,Morales,1976-11-09,48,M,Soltero,...,1832.75,763.53,104986.09,490,Regular,Telemarketing,180,2010/03/12,15,Activo
4,5,DNI,86518506,Sandra,González,Morales,1980/11/20,44,F,Viudo,...,850.00,405.03,37044.25,392,Regular,Digital,145,2013/01/08,4,Activo


In [6]:
# Selecciona filas con datos duplicados 
df_clientes_tra[df_clientes_tra.duplicated(keep=False)].sort_values(by='cliente_id')

,cliente_id,tipo_documento,numero_documento,nombres,apellido_paterno,apellido_materno,fecha_nacimiento,edad,genero,estado_civil,...,ingresos_mensuales,egresos_mensuales,patrimonio_estimado,score_crediticio,segmento_cliente,canal_captacion,antiguedad_cliente_meses,fecha_registro,sucursal_id,estado_cliente
2578,2579,DNI,85980690,Víctor,Rojas,Vargas,24/01/1981,43,M,SOLTERO,...,2119.59,1002.15,56134.8,657,Regular,Digital,83,26-02-2018,17,Activo
5002,2579,DNI,85980690,Víctor,Rojas,Vargas,24/01/1981,43,M,SOLTERO,...,2119.59,1002.15,56134.8,657,Regular,Digital,83,26-02-2018,17,Activo


In [7]:
# Elimina datos duplicados
df_clientes_tra.drop_duplicates(inplace=True)
df_clientes_tra.reset_index(drop=True,inplace=True)

## cliente_id

In [8]:
# Verifica si existen ids negativos o 0 
# Resultados Esperados: Tabla Vacia
df_clientes_tra[df_clientes_tra.cliente_id <= 0]

,cliente_id,tipo_documento,numero_documento,nombres,apellido_paterno,apellido_materno,fecha_nacimiento,edad,genero,estado_civil,...,ingresos_mensuales,egresos_mensuales,patrimonio_estimado,score_crediticio,segmento_cliente,canal_captacion,antiguedad_cliente_meses,fecha_registro,sucursal_id,estado_cliente


In [9]:
# Resultados Esperados: Tabla Vacia 
df_clientes_tra[df_clientes_tra.cliente_id.duplicated(keep=False)]

,cliente_id,tipo_documento,numero_documento,nombres,apellido_paterno,apellido_materno,fecha_nacimiento,edad,genero,estado_civil,...,ingresos_mensuales,egresos_mensuales,patrimonio_estimado,score_crediticio,segmento_cliente,canal_captacion,antiguedad_cliente_meses,fecha_registro,sucursal_id,estado_cliente
250,251,DNI,32538179,Juan,Torres,Medina,1994/07/05,30,M,Conviviente,...,1117.11,439.11,30222.87,433,Regular,Agencia,144,2013-02-10,21,Inactivo
1034,1035,DNI,40369481,Enrique,Chávez,Jiménez,1963/06/23,61,M,Conviviente,...,5565.11,3567.80,217529.53,481,Regular,Digital,150,2012/08/09,24,Inactivo
3525,3526,DNI,99070773,Enrique,Sánchez,Ramírez,18-08-1977,47,M,Soltero,...,5323.07,2449.47,187306.73,494,Regular,Digital,116,2015/06/05,2,Inactivo
3533,3534,DNI,51895539,Miguel,Rodríguez,Cusi,1978/03/12,46,M,Viudo,...,3497.71,1661.86,112419.79,479,Regular,Agencia,179,2010-04-02,18,Activo
5000,1035,DNI,40369481,Enrique,Chávez,Jiménez,1963/06/23,61,M,Conviviente,...,5565.11,3567.80,217529.53,481,Regular,Digital,150,2012-08-09,24,Inactivo
5001,251,DNI,32538179,Juan,Torres,Medina,1994-07-05,30,M,Conviviente,...,1117.11,439.11,30222.87,433,Regular,Agencia,144,2013/02/10,21,Inactivo
5002,3526,DNI,99070773,Enrique,Sánchez,Ramírez,1977-08-18,47,M,Soltero,...,5323.07,2449.47,187306.73,494,Regular,Digital,116,2015-06-05,2,Inactivo
5003,3534,DNI,51895539,Miguel,Rodríguez,Cusi,1978/03/12,46,M,Viudo,...,3497.71,1661.86,112419.79,479,Regular,Agencia,179,2010/04/02,18,Activo


## tipo_documento

In [10]:
# Resultados Esperados: 'DNI', 'CE', 'Pasaporte'
df_clientes_tra.tipo_documento.unique()

array(['DNI', 'CE', 'Pasaporte'], dtype=object)

## numero_documento

**NOTA**: Existe una cantidad considerable de clientes, con numero de documento incorrecto, no tengo muchas ideas de como arreglar este problema la verdad, si fueran datos reales, podría consultar en la RENIEC si existen o no, o con los datos como nombres y apellidos verificar si el numero de documento es el real, tambien si existen registros anteriores podria validarlos.
Para fines practicos voy a separar estos datos incorrectos para revision, y para el proyecto en si, solo los voy a ingresar como **n/a**. 

In [11]:
# Verifica que no existan espacios en blanco 
# Resultados Esperados: Tabla Vacia
df_clientes_tra[df_clientes_tra.numero_documento != df_clientes_tra.numero_documento.str.strip()]

,cliente_id,tipo_documento,numero_documento,nombres,apellido_paterno,apellido_materno,fecha_nacimiento,edad,genero,estado_civil,...,ingresos_mensuales,egresos_mensuales,patrimonio_estimado,score_crediticio,segmento_cliente,canal_captacion,antiguedad_cliente_meses,fecha_registro,sucursal_id,estado_cliente


In [12]:
# Verifica que todos los caracteres sean numeros 
# Resultados Esperados: Tabla Vacia
df_clientes_tra[~(df_clientes_tra.numero_documento.str.isnumeric())]

,cliente_id,tipo_documento,numero_documento,nombres,apellido_paterno,apellido_materno,fecha_nacimiento,edad,genero,estado_civil,...,ingresos_mensuales,egresos_mensuales,patrimonio_estimado,score_crediticio,segmento_cliente,canal_captacion,antiguedad_cliente_meses,fecha_registro,sucursal_id,estado_cliente


In [13]:
# Verifica si la cantidad de digitos son correctos para cada tipo de documento 
# Resultados Esperados: Tabla Vacía 
numero_documento_incorrectos = df_clientes_tra[   
                    # DNIs que no tengan 8 digitos
                    (
                        (df_clientes_tra.tipo_documento == 'DNI') 
                        & ~(df_clientes_tra.numero_documento.str.fullmatch(r"\d{8}"))
                    )
                    # CE que no tengan 9 digitos
                |   (
                        (df_clientes_tra.tipo_documento == 'CE') 
                        & ~(df_clientes_tra.numero_documento.str.fullmatch(r"\d{9}"))
                    )
                    # Pasaporte que tenga menos de 6 digitos y mas de 12 
                    # Por lo que puede investigar si es posible que los pasaportes tengan letras, pero en estos datos solo tienen digitos
                |   (
                        (df_clientes_tra.tipo_documento == "Pasaporte")
                        & ~(df_clientes_tra.numero_documento.str.fullmatch(r"\d{6,12}"))
                    )
                ]

df_clientes_tra.loc[numero_documento_incorrectos.index,['cliente_id','tipo_documento','numero_documento']]

,cliente_id,tipo_documento,numero_documento
8,9,DNI,8427109
17,18,DNI,2895171
46,47,DNI,5486874
56,57,DNI,1687339
64,65,DNI,6724004
...,...,...,...
4909,4910,DNI,3985872
4935,4936,CE,81026926
4959,4960,DNI,1004220
4976,4977,DNI,242459


In [14]:
# Verifica si existen documentos que inicien con 0, esto lo hago principalmente porque, puede existar un 0 a la izquierda por error y por eso el numero es incorrecto, aunque eso es debatible igualmente.
df_prueba = df_clientes_tra.loc[numero_documento_incorrectos.index,['tipo_documento','numero_documento']]
df_prueba[df_prueba.numero_documento.str.startswith("0")]

,tipo_documento,numero_documento


In [15]:
# Separa los registros erroneos
df_revisar_digitos_documento = df_clientes_tra.loc[numero_documento_incorrectos.index,["cliente_id","tipo_documento","numero_documento"]].copy()
df_revisar_digitos_documento["ERROR"] = "CANTIDAD DE DIGITOS INCORRECTOS PARA EL TIPO DE DOCUMENTO"

In [16]:
# Reemplaza los numeros incorrectos por 'n/a' 
df_clientes_tra.loc[numero_documento_incorrectos.index,'numero_documento'] = 'n/a'
df_clientes_tra.loc[numero_documento_incorrectos.index,'numero_documento']

8       n/a
17      n/a
46      n/a
56      n/a
64      n/a
       ... 
4909    n/a
4935    n/a
4959    n/a
4976    n/a
4981    n/a
Name: numero_documento, Length: 499, dtype: object

## nombres

In [17]:
# Nombres que tienen formatos incorrectos 
# Resultados Esperados Tabla Vacia
df_clientes_tra.nombres[df_clientes_tra.nombres != df_clientes_tra.nombres.str.strip().str.title()]

6           lucía
10         ÓsCaR 
29         ANDRÉS
42       daniela 
72         Víctor
          ...    
4900      Valeria
4924     Eduardo 
4951         luis
4978     PATRICIA
4979      roberto
Name: nombres, Length: 288, dtype: object

In [18]:
# Limpieza de los nombres 
df_clientes_tra['nombres'] = df_clientes_tra.nombres.apply(limpiar_texto)
# Nombres que tienen formatos incorrectos 
# Resultados Esperados Tabla Vacia
df_clientes_tra.nombres[df_clientes_tra.nombres != df_clientes_tra.nombres.str.strip().str.title()]

Series([], Name: nombres, dtype: object)

## apellido_paterno

In [19]:
# Apellidos que tienen formatos incorrectos
# Resultados Esperados Tabla Vacia
df_clientes_tra.apellido_paterno[df_clientes_tra.apellido_paterno 
                                != df_clientes_tra.apellido_paterno.str.strip().str.title()]

16      CASTILLO 
27         MamAni
65       ramírez 
69          OrtIz
90         CaStRo
          ...    
4933      Ramos  
4941       mamani
4942     Quispe  
4988      Sánchez
5002     Sánchez 
Name: apellido_paterno, Length: 283, dtype: object

In [20]:
# Limpieza de los apellidos paternos 
df_clientes_tra['apellido_paterno'] = df_clientes_tra.apellido_paterno.apply(limpiar_texto)
# Apellidos que tienen formatos incorrectos
# Resultados Esperados Tabla Vacia
df_clientes_tra.apellido_paterno[df_clientes_tra.apellido_paterno 
                                != df_clientes_tra.apellido_paterno.str.strip().str.title()]

Series([], Name: apellido_paterno, dtype: object)

## apellido_materno

In [21]:
# Apellidos que tienen formatos incorrectos
# Resultados Esperados Tabla Vacia
df_clientes_tra.apellido_materno[df_clientes_tra.apellido_materno 
                                != df_clientes_tra.apellido_materno.str.strip().str.title()]

6         Reyes  
15           cruz
16       Condori 
21          lópez
23          OrTiZ
          ...    
4898      MedIna 
4908     Chávez  
4946    martínez 
4974       Quispe
4997     Mamani  
Name: apellido_materno, Length: 283, dtype: object

In [22]:
# Limpieza de los apellidos materno 
df_clientes_tra['apellido_materno'] = df_clientes_tra.apellido_materno.apply(limpiar_texto)
# Apellidos que tienen formatos incorrectos
# Resultados Esperados Tabla Vacia
df_clientes_tra.apellido_materno[df_clientes_tra.apellido_materno 
                                != df_clientes_tra.apellido_materno.str.strip().str.title()]

Series([], Name: apellido_materno, dtype: object)

## fecha_nacimiento

In [23]:
# Verifica si existen fechas con errores 
fechas_error_nacimiento = pd.to_datetime(
    df_clientes_tra.fecha_nacimiento,
    format='%Y-%m-%d', 
    errors='coerce'
)
# Muestras las fechas que generan error 
# Resultados Esperados: Tabla Vacia 
df_clientes_tra.fecha_nacimiento[fechas_error_nacimiento.isna()]

0       16/09/1977
1       13/03/1964
4       1980/11/20
8       1989/10/02
9       21/06/1978
           ...    
4995    24/07/1982
4997    28-07-1972
4998    1987/01/05
5000    1963/06/23
5003    1978/03/12
Name: fecha_nacimiento, Length: 3304, dtype: object

In [24]:
# limpia las fechas 
df_clientes_tra['fecha_nacimiento'] = pd.to_datetime(
        df_clientes_tra.fecha_nacimiento,
        format="mixed",
        errors='coerce',
        dayfirst=True
    )
df_clientes_tra.fecha_nacimiento[fechas_error_nacimiento.isna()]

0      1977-09-16
1      1964-03-13
4      1980-11-20
8      1989-10-02
9      1978-06-21
          ...    
4995   1982-07-24
4997   1972-07-28
4998   1987-01-05
5000   1963-06-23
5003   1978-03-12
Name: fecha_nacimiento, Length: 3304, dtype: datetime64[ns]

In [25]:
# Verifica si existen fechas de nacimiento futuras o fechas muy antiguas 
# Resultados Esperados: Tabla Vacia 
df_clientes_tra.fecha_nacimiento[
    (df_clientes_tra.fecha_nacimiento.dt.date >= date.today())
    | (df_clientes_tra.fecha_nacimiento <= pd.to_datetime('1900-01-01',format='%Y-%m-%d'))
]

Series([], Name: fecha_nacimiento, dtype: datetime64[ns])

## edad

In [26]:
# Recalcula la edad, ya que es un dato que esta desactualizado 
# Realmente no se tienen que guardar edad, salvo casos especificos como, edad desde fecha de corte, pero bueno, para esta capa plata aun va aparecer, luego en la capa oro se va a ir, porque, es un valor derivado de otra columna igual que antiguedad_cliente_meses 
fecha_actual = date.today()
df_clientes_tra["edad"] = (
    fecha_actual.year - df_clientes_tra.fecha_nacimiento.dt.year
    - (
        (fecha_actual.month < df_clientes_tra.fecha_nacimiento.dt.month)
        |
        (
            (fecha_actual.month == df_clientes_tra.fecha_nacimiento.dt.month)
            &
            (fecha_actual.day <= df_clientes_tra.fecha_nacimiento.dt.day)
        )
    )
)
df_clientes_tra.edad[df_clientes_tra.edad.isna()]

Series([], Name: edad, dtype: int32)

## genero

In [27]:
# Muestra los valores unicos de genero
# Resultados Esperados: "FEMENINO", "MASCULINO", "n/a"
df_clientes_tra.genero.unique()

array(['F', 'f', 'M', 'FEMENINO', 'm', 'fem', 'mas', 'Femenino', 'M ',
       'Masculino', 'MASCULINO', 'F '], dtype=object)

In [28]:
# Resultados Esperados: "FEMENINO", "MASCULINO", "n/a"
df_clientes_tra["genero"] = df_clientes_tra.genero.apply(formato_genero)
df_clientes_tra.genero.unique()

array(['Femenino', 'Masculino'], dtype=object)

## estado_civil

In [29]:
# Resultados Esperados: 'Casado', 'Soltero', 'Viudo', 'Divorciado', 'Conviviente', 'n/a'
df_clientes_tra.estado_civil.unique()

array(['Casado', 'Soltero', 'Viudo', 'Divorciado', 'Conviviente', 'viudo',
       'soltero', 'Casado ', 'Soltero ', 'casado', 'casado ',
       'DIVORCIADO', 'CASADO', 'SOLTERO', 'Divorciado ', 'soltero ',
       'Viudo ', 'divorciado', 'VIUDO'], dtype=object)

In [30]:
# Aplica el formato de texto adecuado a estado civil
# Resultados Esperados: 'Casado', 'Soltero', 'Viudo', 'Divorciado', 'Conviviente', 'n/a'
df_clientes_tra["estado_civil"] = df_clientes_tra.estado_civil.apply(limpiar_texto)
df_clientes_tra.estado_civil.unique()

array(['Casado', 'Soltero', 'Viudo', 'Divorciado', 'Conviviente'],
      dtype=object)

## nivel_educacion

In [31]:
# Resultados Esperados: 'Secundaria', 'Universitario', 'Tecnico', 'Postgrado', 'n/a'
df_clientes_tra.nivel_educacion.unique()

array(['Secundaria', 'Universitario', 'Técnico', 'Postgrado', None,
       'universitario', 'tecnico', 'Tecnico', 'UNIVERSITARIO',
       'Universitario ', 'universitario ', 'técnico', 'secundaria',
       'SECUNDARIA', 'secundaria ', 'TECNICO'], dtype=object)

In [32]:
# Aplica el formato de texto adecuado a nivel_educacion
# Resultados Esperados: 'Secundaria', 'Universitario', 'Tecnico', 'Postgrado', 'n/a'
df_clientes_tra["nivel_educacion"] = df_clientes_tra.nivel_educacion.apply(formato_nivel_educacion)
df_clientes_tra.nivel_educacion.unique()

array(['Secundaria', 'Universitario', 'Técnico', 'Postgrado', 'n/a'],
      dtype=object)

## ocupacion

In [33]:
# Resultados esperados: 'Empleado Público', 'Empresario', 'Empleado Privado', 'Independiente', 'Pensionista',    'Estudiante', 'n/a'
df_clientes_tra.ocupacion.unique()

array(['Empleado Público', 'Empresario', 'Empleado Privado',
       'Independiente', 'Pensionista', 'Estudiante', None], dtype=object)

In [34]:
# Resultados esperados: 'Empleado Público', 'Empresario', 'Empleado Privado', 'Independiente',      'Pensionista',    'Estudiante', 'n/a'
df_clientes_tra['ocupacion'] = df_clientes_tra.ocupacion.apply(limpiar_texto)
df_clientes_tra.ocupacion.unique()

array(['Empleado Público', 'Empresario', 'Empleado Privado',
       'Independiente', 'Pensionista', 'Estudiante', 'n/a'], dtype=object)

## sector_economico

In [35]:
# Resultados Esperados: 'Transporte', 'Servicios', 'Otro', 'Comercio', 'Manufactura', 'Tecnología', 'Salud', 'Finanzas','Educación', 'Construcción', 'Agricultura'
df_clientes_tra.sector_economico.unique()

array(['Transporte', 'Servicios', 'Otro', 'Comercio', 'Manufactura',
       'Tecnología', 'Salud', 'Finanzas', 'Educación', 'Construcción',
       'Agricultura'], dtype=object)

## ciudad_residencia

In [36]:
# Verifica el formato de ciudad_residencia
# Resultados Esperados: Tabla Vacia
df_clientes_tra.ciudad_residencia[df_clientes_tra.ciudad_residencia 
                                != df_clientes_tra.ciudad_residencia.str.strip().str.title()]

25           Santiago de Surco
32                La esperanza
37           Santiago de Surco
59      San Juan de Lurigancho
60                    Sullana 
                 ...          
4965    San Juan de Lurigancho
4976                 Chiclayo 
4996                 ChIcLaYo 
5001                  chimbote
5002         Santiago de Surco
Name: ciudad_residencia, Length: 677, dtype: object

In [37]:
# Aplica el formato de texto adecuado a ciudad_residencia
df_clientes_tra["ciudad_residencia"] = df_clientes_tra.ciudad_residencia.apply(limpiar_texto)
# Verifica el formato de ciudad_residencia
# Resultados Esperados: Santiago de Surco, San Juan de Lurigancho
df_clientes_tra.ciudad_residencia[df_clientes_tra.ciudad_residencia 
                                != df_clientes_tra.ciudad_residencia.str.strip().str.title()].unique()

array(['Santiago de Surco', 'San Juan de Lurigancho'], dtype=object)

In [38]:
# Valida si todas las ciudades de residencia de clientes, se encuentra en el registro de la empresa 
# Resultados Esperados: both: 5004, left_only: 0, right_only: 0
verificar_ciudad = df_clientes_tra.merge(
    right=df_ciudades_tra,
    right_on='ciudad',
    left_on='ciudad_residencia',
    indicator=True)

verificar_ciudad._merge.value_counts()

_merge
both          5004
left_only        0
right_only       0
Name: count, dtype: int64

## departamento_residencia

In [39]:
# Verifica el formato de departamento_residencia
# Resultados Esperados: Tabla Vacia
df_clientes_tra.departamento_residencia[df_clientes_tra.departamento_residencia 
                                != df_clientes_tra.departamento_residencia.str.strip().str.title()]

27              Ica 
39              ica 
50          arequipa
101         arequipa
131           Junín 
            ...     
4924    La Libertad 
4946       Arequipa 
4950     LA LIBERTAD
4971      Lambayeque
4994         CUSCO  
Name: departamento_residencia, Length: 270, dtype: object

In [40]:
# Aplica el formato de texto adecuado a departamento_residencia
df_clientes_tra["departamento_residencia"] = df_clientes_tra.departamento_residencia.apply(limpiar_texto)
# Resultados Esperados: Tabla Vacia
df_clientes_tra.departamento_residencia[df_clientes_tra.departamento_residencia 
                                != df_clientes_tra.departamento_residencia.str.strip().str.title()]

Series([], Name: departamento_residencia, dtype: object)

In [41]:
# Valida si todas los departamentos de residencia de los clientes, se encuentran en el registro de la empresa 
# Resultados Esperados: Tabla Vacia 
verificar_departamento = df_clientes_tra.merge(
    right=df_ciudades_tra,
    right_on='ciudad',
    left_on='ciudad_residencia'
    )

verificar_departamento[verificar_departamento.departamento_residencia != verificar_departamento.departamento].departamento.unique()

array(['Áncash'], dtype=object)

In [42]:
df_clientes_tra.loc[df_clientes_tra.departamento_residencia == "Ancash","departamento_residencia"] = "Áncash"

## region_residencia 

In [43]:
# Verifica el formato de ciudad_residencia
# Resultados esperados: Lima y Callao
df_clientes_tra.region_residencia[df_clientes_tra.region_residencia 
                                != df_clientes_tra.region_residencia.str.strip().str.title()]

4       Lima y Callao
9       Lima y Callao
13      Lima y Callao
19      Lima y Callao
25      Lima y Callao
            ...      
4984    Lima y Callao
4987    Lima y Callao
4988    Lima y Callao
4995    Lima y Callao
5002    Lima y Callao
Name: region_residencia, Length: 1031, dtype: object

In [44]:
# Aplica el formato de texto adecuado a region_residencia
df_clientes_tra["region_residencia"] = df_clientes_tra.region_residencia.apply(limpiar_texto)
# Verifica el formato de ciudad_residencia
# Resultados esperados: Lima y Callao
df_clientes_tra.region_residencia[df_clientes_tra.region_residencia 
                                != df_clientes_tra.region_residencia.str.strip().str.title()]

4       Lima y Callao
9       Lima y Callao
13      Lima y Callao
19      Lima y Callao
25      Lima y Callao
            ...      
4984    Lima y Callao
4987    Lima y Callao
4988    Lima y Callao
4995    Lima y Callao
5002    Lima y Callao
Name: region_residencia, Length: 1031, dtype: object

In [45]:
# Valida si todas las regiones de residencia de los clientes, se encuentran en el registro de la empresa 
# Resultados Esperados: Tabla Vacia 
verificar_region = df_clientes_tra.merge(
    right=df_ciudades_tra,
    right_on='ciudad',
    left_on='ciudad_residencia'
    )

verificar_region[verificar_region.region_residencia != verificar_region.region]

,cliente_id,tipo_documento,numero_documento,nombres,apellido_paterno,apellido_materno,fecha_nacimiento,edad,genero,estado_civil,...,score_crediticio,segmento_cliente,canal_captacion,antiguedad_cliente_meses,fecha_registro,sucursal_id,estado_cliente,ciudad,departamento,region


## ingresos_mensuales

In [46]:
# Verifica si existen ingresos_mensuales mmenores o iguales a 0
# Resultados esperados: Tabla Vacia 
df_clientes_tra.ingresos_mensuales[df_clientes_tra.ingresos_mensuales <= 0]

0       -2072.19
39      -1467.02
47      -1235.37
151     -2284.29
423    -12210.84
652    -11396.26
661     -4934.84
739    -10063.69
777     -1323.38
1001    -4389.26
1097    -4494.68
1132    -6269.78
1191    -1532.25
1195    -1767.69
1213    -1709.26
1279    -1220.13
1368    -3192.14
1491    -1536.50
1604    -1398.84
1638    -2376.19
1725    -1086.82
1756    -3564.17
1864    -2817.39
2344    -1822.16
2421    -2967.07
2533    -5429.68
2769    -2411.63
2917    -4058.15
2985    -3114.23
3169    -1904.26
3232    -3613.78
3337    -1587.02
3601    -1520.36
3751    -1981.21
3784    -1296.23
3854    -3554.02
3897    -6933.15
3965    -1807.38
4107    -1130.76
4143    -1743.14
4289    -2591.12
4332    -1599.95
4343    -3759.38
4460    -5876.52
4476    -2202.07
4491    -1260.66
4551    -3526.20
4659    -2488.22
4660    -2385.83
4765    -1898.44
4791    -3346.72
4826    -3690.51
Name: ingresos_mensuales, dtype: float64

In [47]:
# Aplica valor absoluto 
df_clientes_tra["ingresos_mensuales"] = df_clientes_tra.ingresos_mensuales.apply(abs)
# Verifica si existen ingresos mensuales mmenores o iguales a 0
# Resultados esperados: Tabla Vacia 
df_clientes_tra.ingresos_mensuales[df_clientes_tra.ingresos_mensuales <= 0]

Series([], Name: ingresos_mensuales, dtype: float64)

## egresos_mensuales 

In [48]:
# Verifica si existen egresos_mensuales mmenores o iguales a 0
# Resultados esperados: Tabla Vacia
df_clientes_tra.egresos_mensuales[df_clientes_tra.egresos_mensuales <= 0 ]

10      -4606.94
68       -753.02
98      -1229.94
176      -985.66
331     -1537.49
445     -1088.14
588      -718.22
622     -2762.85
726     -1681.51
747     -4490.66
835     -7794.18
1046    -1105.20
1418    -3333.69
1567    -1928.76
1662    -1463.21
1745    -1122.98
2004    -1926.99
2384    -2049.73
2440    -1375.45
2492    -1399.65
2784    -2120.14
2937     -476.92
2979    -2911.65
3235    -2105.68
3281     -903.92
3302   -16701.67
3372    -4846.98
3374     -736.35
3379    -5088.67
3507     -463.74
3604    -1095.44
3656    -1234.32
3667    -2613.82
3672    -1238.88
3721    -3324.13
3923     -370.88
3981    -5924.99
4008    -1581.19
4160     -837.13
4169     -776.94
4172    -2832.76
4341    -1515.63
4367    -1153.72
4462    -2283.54
4486    -1190.44
4626     -673.58
4677    -2016.26
4680    -2564.10
4998    -1063.05
Name: egresos_mensuales, dtype: float64

In [49]:
# Aplica valor absoluto 
df_clientes_tra["egresos_mensuales"] = df_clientes_tra.egresos_mensuales.apply(abs)
# Verifica si existen egresos_mensuales mmenores o iguales a 0 o que los egresos sean mayores o iguales a los ingresos mensuales 
# Resultados esperados: Tabla Vacia 
df_clientes_tra.egresos_mensuales[(df_clientes_tra.egresos_mensuales <= 0) | (df_clientes_tra.egresos_mensuales >= df_clientes_tra.ingresos_mensuales)]

Series([], Name: egresos_mensuales, dtype: float64)

## patrimonio_estimado

In [50]:
# Verifica si existen valores en patrimonio_estimado menores o iguales a 0 
# Resultados Esperados: Tabla Vacia 
df_clientes_tra.patrimonio_estimado[df_clientes_tra.patrimonio_estimado <= 0]

Series([], Name: patrimonio_estimado, dtype: float64)

## score_crediticio

In [51]:
# Verifica si existen valores en score_crediticio que sean menores o iguales a 0 o que sean mayores a 1000 
# Resultados Esperados: Tabla Vacia 
df_clientes_tra.score_crediticio[(df_clientes_tra.score_crediticio <= 0) | (df_clientes_tra.score_crediticio > 1000)]

395     1050
596       -5
635       -5
650     1050
967       -5
1014    1050
1132      -5
1161    1200
1447    1200
2214    1050
2555      -5
2728      -5
2896    1050
2909    1200
3328    1050
3612    1200
3631    1200
4253      -5
4270      -5
4698    1200
4700      -5
4905    1200
Name: score_crediticio, dtype: int64

In [52]:
# Aplica valor absoluto para eliminar nulos 
df_clientes_tra["score_crediticio"] = df_clientes_tra.score_crediticio.apply(abs)
# Limitando los scores crediticios a 1000  
df_clientes_tra["score_crediticio"] = df_clientes_tra.score_crediticio.apply(
    lambda score: 1000 if score >1000 else score
)
# Verifica si existen valores en score_crediticio que sean menores o iguales a 0 o que sean mayores a 1000 
# Resultados Esperados: Tabla Vacía 
df_clientes_tra.score_crediticio[(df_clientes_tra.score_crediticio <= 0) | (df_clientes_tra.score_crediticio > 1000)]

Series([], Name: score_crediticio, dtype: int64)

## segmento_cliente

In [53]:
# Muestra los valores unicos
# Resultados esperados: 'Regular', 'Premium', 'Vip'
df_clientes_tra.segmento_cliente.unique()

array(['Regular', 'regular', 'Premium', 'Premium ', 'regular ', 'REGULAR',
       'VIP', 'PREMIUM', 'VIP ', 'premium'], dtype=object)

In [54]:
# Estandariza los valores de segmento_cliente 
# Resultados esperados: 'Regular', 'Premium', 'Vip'
df_clientes_tra["segmento_cliente"] = df_clientes_tra.segmento_cliente.apply(limpiar_texto)
df_clientes_tra.loc[df_clientes_tra.segmento_cliente == "Vip","segmento_cliente"] = df_clientes_tra.loc[df_clientes_tra.segmento_cliente == "Vip","segmento_cliente"].str.upper()
df_clientes_tra.segmento_cliente.unique()

array(['Regular', 'Premium', 'VIP'], dtype=object)

## canal_captacion

In [55]:
# Muestra los valores unicos
# Resultados Esperados:'Agencia', 'Digital', 'Telemarketing', 'Referido', 'Campaña Institucional', 'n/a'
df_clientes_tra.canal_captacion.unique()

array(['Agencia', 'Digital', 'Telemarketing', 'Referido',
       'Campaña Institucional', None], dtype=object)

In [56]:
# Estandariza los valores de canal_captacion
# Resultados Esperados:'Agencia', 'Digital', 'Telemarketing', 'Referido', 'Campaña Institucional', 'n/a'
df_clientes_tra["canal_captacion"] = df_clientes_tra.canal_captacion.apply(limpiar_texto)
df_clientes_tra.canal_captacion.unique()

array(['Agencia', 'Digital', 'Telemarketing', 'Referido',
       'Campaña Institucional', 'n/a'], dtype=object)

## fecha_registro 

In [57]:
# Verifica si existen fechas con formatos incorrectos
# Resultados Esperados: Tabla Vacia 
fechas_error_registro = pd.to_datetime(df_clientes_tra.fecha_registro, errors='coerce')
df_clientes_tra.fecha_registro[(fechas_error_registro.isna())]

1       2017-10-18
5       2015-02-09
6       14-02-2019
7       19-06-2017
8       2013-09-26
           ...    
4992    29-02-2020
4993    2016-01-24
4998    2020-11-22
5000    2012-08-09
5002    2015-06-05
Name: fecha_registro, Length: 3227, dtype: object

In [58]:
# Transforma las fechas utilizando format = 'mixed'
df_clientes_tra["fecha_registro"] = pd.to_datetime(
    df_clientes_tra.fecha_registro, 
    errors='coerce',
    format='mixed',
    dayfirst=True
)
# Muestra los registros que antes generaban error
# Resultados Esperados: Tabla Vacia 
df_clientes_tra.fecha_registro[(fechas_error_registro.isna())]

1      2017-10-18
5      2015-02-09
6      2019-02-14
7      2017-06-19
8      2013-09-26
          ...    
4992   2020-02-29
4993   2016-01-24
4998   2020-11-22
5000   2012-08-09
5002   2015-06-05
Name: fecha_registro, Length: 3227, dtype: datetime64[ns]

**Nota**: Existe una fecha de registro futura, pero, no tengo otro registro que me pueda ayudar, lo que voy hacer es utilizar la tabla de prestamos para limpiar estas fechas, la limpieza se realiza en 07_validacion_clientes_prestamos.ipynb.

In [59]:
# Verifica si existen fechas de registro futuras 
# Resultados Esperados: Tabla Vacia 
df_revisar_fechas_futuras = df_clientes_tra[['cliente_id','fecha_nacimiento','edad','fecha_registro']][
    df_clientes_tra.fecha_registro.dt.date >= date.today()
]
df_revisar_fechas_futuras

,cliente_id,fecha_nacimiento,edad,fecha_registro
1361,1362,2014-07-20,12,2032-07-20


In [60]:
df_revisar_fechas_futuras["ERROR"] = "FECHA DE REGISTRO FUTURA"

In [61]:
# Verifica si existen registros donde la fecha_nacimiento sea mayor o igual a la fecha_registro
# Resultados Esperados: Tabla Vacia 
df_clientes_tra.fecha_registro[df_clientes_tra.fecha_nacimiento >= df_clientes_tra.fecha_registro]

Series([], Name: fecha_registro, dtype: datetime64[ns])

In [62]:
# Verifica que la fecha de registro sea cuando el cliente tiene 18 años o más 
prueba_fechas = df_clientes_tra.copy()
prueba_fechas["edad_registro"] = (
    prueba_fechas.fecha_registro.dt.year
    - prueba_fechas.fecha_nacimiento.dt.year
    - ( 
        (prueba_fechas.fecha_registro.dt.month < prueba_fechas.fecha_nacimiento.dt.month)
        |
        (
            (prueba_fechas.fecha_registro.dt.month == prueba_fechas.fecha_nacimiento.dt.month)
            &
            (prueba_fechas.fecha_registro.dt.day < prueba_fechas.fecha_nacimiento.dt.day)
        )
    )
)

# Selecciona los registros con fechas erroneas 
# Resultados Esperados: Tabla Vacia
prueba_fechas = prueba_fechas[prueba_fechas.edad_registro <18]
prueba_fechas[["cliente_id","fecha_nacimiento","fecha_registro","edad_registro"]]

,cliente_id,fecha_nacimiento,fecha_registro,edad_registro
11,12,1995-01-22,2010-12-20,15
21,22,1997-03-03,2012-02-13,14
68,69,1999-04-08,2010-08-11,11
137,138,1996-08-22,2014-03-06,17
143,144,2004-03-26,2011-07-04,7
...,...,...,...,...
4934,4935,2004-12-29,2014-03-10,9
4940,4941,2000-11-09,2016-10-15,15
4950,4951,2004-01-21,2021-04-15,17
4976,4977,2002-10-28,2012-05-07,9


- Para este caso me tomo el atrevimiento de modificar las fechas de nacimiento, realmente en un entorno real esta información se tiene que separar para evaluar de manera independiente. 
- Estos registros incorrectos se van almacenar de manera independiente para comprarse luego.  

In [63]:
# Selecciona los registros incorrectos y marca el error encontrado
df_revisar_fechas_registro = prueba_fechas[["cliente_id","fecha_nacimiento","fecha_registro","edad_registro"]].copy()
df_revisar_fechas_registro['ERROR'] = "EDAD DE REGISTRO MENOR A 18 AÑOS"

In [64]:
# La nueva fecha de nacimiento = fecha de registro - 18 años - 1 dia
prueba_fechas["fecha_nacimiento_correcta"] = prueba_fechas.fecha_registro - pd.DateOffset(years= 18, days= 1) 
prueba_fechas[["cliente_id","fecha_nacimiento","fecha_registro","fecha_nacimiento_correcta","edad_registro"]]

,cliente_id,fecha_nacimiento,fecha_registro,fecha_nacimiento_correcta,edad_registro
11,12,1995-01-22,2010-12-20,1992-12-19,15
21,22,1997-03-03,2012-02-13,1994-02-12,14
68,69,1999-04-08,2010-08-11,1992-08-10,11
137,138,1996-08-22,2014-03-06,1996-03-05,17
143,144,2004-03-26,2011-07-04,1993-07-03,7
...,...,...,...,...,...
4934,4935,2004-12-29,2014-03-10,1996-03-09,9
4940,4941,2000-11-09,2016-10-15,1998-10-14,15
4950,4951,2004-01-21,2021-04-15,2003-04-14,17
4976,4977,2002-10-28,2012-05-07,1994-05-06,9


In [65]:
df_clientes_tra = df_clientes_tra.merge(
    prueba_fechas[["cliente_id", "fecha_nacimiento_correcta"]],
    on="cliente_id",
    how="left"
)
# Cambia los datos erroneos por el nuevo recalculo 
df_clientes_tra["fecha_nacimiento"] = (
    df_clientes_tra["fecha_nacimiento_correcta"]
    .fillna(df_clientes_tra["fecha_nacimiento"])
)

df_clientes_tra.drop(columns="fecha_nacimiento_correcta", inplace=True)

**Nota**: Todavia se tiene que verificar que el cliente se registro despues de la apertura de la sucursal, la limpieza se va a realizar en [sucursal_id](#sucursal_id) 

## edad V2

In [66]:
# Recalcula la edad, luego de cambiar la fecha de nacimiento
fecha_actual = date.today()
df_clientes_tra["edad"] = (
    fecha_actual.year - df_clientes_tra.fecha_nacimiento.dt.year
    - (
        (fecha_actual.month < df_clientes_tra.fecha_nacimiento.dt.month)
        |
        (
            (fecha_actual.month == df_clientes_tra.fecha_nacimiento.dt.month)
            &
            (fecha_actual.day < df_clientes_tra.fecha_nacimiento.dt.day)
        )
    )
)
df_clientes_tra.edad[df_clientes_tra.edad.isna()]

Series([], Name: edad, dtype: int32)

## antiguedad_cliente_meses

In [67]:
# Recalcula la antiguedad de los clientes en meses  
fecha_actual = date.today()

df_clientes_tra['antiguedad_cliente_meses'] = (
    (fecha_actual.year - df_clientes_tra.fecha_registro.dt.year) * 12
    + (fecha_actual.month - df_clientes_tra.fecha_registro.dt.month)
    - ( fecha_actual.day < df_clientes_tra.fecha_registro.dt.day)
)

df_clientes_tra.antiguedad_cliente_meses[df_clientes_tra.antiguedad_cliente_meses.isna()]

Series([], Name: antiguedad_cliente_meses, dtype: int32)

## sucursal_id

In [68]:
# Resultados Esperados: Tabla Vacia
df_clientes_tra.sucursal_id[df_clientes_tra.sucursal_id <= 0]

Series([], Name: sucursal_id, dtype: int64)

In [69]:
# Verifica que todas las sucursales esten dentro de la tabla de sucursales 
# Resultados Esperados: both: 5004, left_only: 0, right_only: 0
df_clientes_veri = df_clientes_tra[['cliente_id','sucursal_id','fecha_registro']].copy()
df_sucursales_veri = df_sucursales_tra[['sucursal_id','fecha_apertura']]
df_merge_veri_sucursal = df_clientes_veri.merge(
    right=df_sucursales_veri,
    on="sucursal_id",
    how='left',
    indicator=True)
df_merge_veri_sucursal._merge.value_counts()

_merge
both          5004
left_only        0
right_only       0
Name: count, dtype: int64

**Nota**: En un entorno real antes de ver cual es la correcta tendria que revisar registros, para este caso, y como igualmente voy a tener que modificara uno de los 2 registros, entonces, lo que voy hacer es modificar fecha de apertura, porque si modifico, fecha_registro, puedo dañar los registros de prestamos y pagos, la modificacion se va a realizar en 04_validacion_clientes_sucursales.  

In [70]:
# Selecciona los registros con errores 
df_revisar_fecha_registro_fecha_sucursal = df_merge_veri_sucursal[["cliente_id","fecha_registro","sucursal_id","fecha_apertura"]][df_merge_veri_sucursal.fecha_registro < df_merge_veri_sucursal.fecha_apertura].copy()
df_revisar_fecha_registro_fecha_sucursal

,cliente_id,fecha_registro,sucursal_id,fecha_apertura
3,4,2010-03-12,15,2010-12-07
4,5,2013-01-08,4,2014-06-19
5,6,2015-02-09,16,2015-04-22
11,12,2010-12-20,19,2013-02-10
16,17,2012-09-14,19,2013-02-10
...,...,...,...,...
4976,4977,2012-05-07,13,2013-07-10
4985,4986,2013-09-29,10,2014-06-25
4996,4997,2013-04-08,13,2013-07-10
5000,1035,2012-08-09,24,2013-07-11


In [71]:
df_revisar_fecha_registro_fecha_sucursal["ERROR"] = "FECHA DE REGISTRO ANTERIOR A FECHA DE APERTURA DE LA SUCURSAL"

## estado_cliente

In [72]:
# Resultados Esperados: 'Activo', 'Inactivo', 'Bloqueado'
# No requiere cambios
df_clientes_tra.estado_cliente.unique()

array(['Activo', 'Inactivo', 'Bloqueado'], dtype=object)

# Limpiando Duplicados Luego de Limpieza

In [73]:
df_clientes_tra[df_clientes_tra.duplicated(keep=False)]

,cliente_id,tipo_documento,numero_documento,nombres,apellido_paterno,apellido_materno,fecha_nacimiento,edad,genero,estado_civil,...,ingresos_mensuales,egresos_mensuales,patrimonio_estimado,score_crediticio,segmento_cliente,canal_captacion,antiguedad_cliente_meses,fecha_registro,sucursal_id,estado_cliente
250,251,DNI,32538179,Juan,Torres,Medina,1994-07-05,32,Masculino,Conviviente,...,1117.11,439.11,30222.87,433,Regular,Agencia,161,2013-02-10,21,Inactivo
1034,1035,DNI,40369481,Enrique,Chávez,Jiménez,1963-06-23,63,Masculino,Conviviente,...,5565.11,3567.80,217529.53,481,Regular,Digital,167,2012-08-09,24,Inactivo
3525,3526,DNI,99070773,Enrique,Sánchez,Ramírez,1977-08-18,48,Masculino,Soltero,...,5323.07,2449.47,187306.73,494,Regular,Digital,133,2015-06-05,2,Inactivo
3533,3534,DNI,51895539,Miguel,Rodríguez,Cusi,1978-03-12,48,Masculino,Viudo,...,3497.71,1661.86,112419.79,479,Regular,Agencia,196,2010-04-02,18,Activo
5000,1035,DNI,40369481,Enrique,Chávez,Jiménez,1963-06-23,63,Masculino,Conviviente,...,5565.11,3567.80,217529.53,481,Regular,Digital,167,2012-08-09,24,Inactivo
5001,251,DNI,32538179,Juan,Torres,Medina,1994-07-05,32,Masculino,Conviviente,...,1117.11,439.11,30222.87,433,Regular,Agencia,161,2013-02-10,21,Inactivo
5002,3526,DNI,99070773,Enrique,Sánchez,Ramírez,1977-08-18,48,Masculino,Soltero,...,5323.07,2449.47,187306.73,494,Regular,Digital,133,2015-06-05,2,Inactivo
5003,3534,DNI,51895539,Miguel,Rodríguez,Cusi,1978-03-12,48,Masculino,Viudo,...,3497.71,1661.86,112419.79,479,Regular,Agencia,196,2010-04-02,18,Activo


In [74]:
# Eliminando duplicados 
df_clientes_tra.drop_duplicates(inplace=True)

## cliente_id V2

In [75]:
# Muestra los registros con id duplicados 
df_clientes_tra[df_clientes_tra.cliente_id.duplicated(keep=False)]

,cliente_id,tipo_documento,numero_documento,nombres,apellido_paterno,apellido_materno,fecha_nacimiento,edad,genero,estado_civil,...,ingresos_mensuales,egresos_mensuales,patrimonio_estimado,score_crediticio,segmento_cliente,canal_captacion,antiguedad_cliente_meses,fecha_registro,sucursal_id,estado_cliente


# Exportando la Tabla Limpia

In [76]:
df_clientes_tra.info()

<class 'pandas.core.frame.DataFrame'>
Index: 5000 entries, 0 to 4999
Data columns (total 26 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   cliente_id                5000 non-null   int64         
 1   tipo_documento            5000 non-null   object        
 2   numero_documento          5000 non-null   object        
 3   nombres                   5000 non-null   object        
 4   apellido_paterno          5000 non-null   object        
 5   apellido_materno          5000 non-null   object        
 6   fecha_nacimiento          5000 non-null   datetime64[ns]
 7   edad                      5000 non-null   int32         
 8   genero                    5000 non-null   object        
 9   estado_civil              5000 non-null   object        
 10  nivel_educacion           5000 non-null   object        
 11  ocupacion                 5000 non-null   object        
 12  sector_economico         

In [77]:
df_clientes_tra.head()

,cliente_id,tipo_documento,numero_documento,nombres,apellido_paterno,apellido_materno,fecha_nacimiento,edad,genero,estado_civil,...,ingresos_mensuales,egresos_mensuales,patrimonio_estimado,score_crediticio,segmento_cliente,canal_captacion,antiguedad_cliente_meses,fecha_registro,sucursal_id,estado_cliente
0,1,DNI,60366909,Carmen,Ramírez,Flores,1977-09-16,48,Femenino,Casado,...,2072.19,888.76,39631.77,550,Regular,Agencia,122,2016-05-07,14,Activo
1,2,DNI,62729806,Cecilia,Cusi,Cusi,1964-03-13,62,Femenino,Casado,...,5663.68,3037.05,125691.99,528,Regular,Digital,105,2017-10-18,13,Activo
2,3,CE,641708053,Juan,Ortiz,Medina,1964-08-01,62,Masculino,Casado,...,4750.66,2302.13,226055.14,493,Regular,Agencia,90,2019-01-08,13,Activo
3,4,DNI,29912419,Carlos,Flores,Morales,1976-11-09,49,Masculino,Soltero,...,1832.75,763.53,104986.09,490,Regular,Telemarketing,196,2010-03-12,15,Activo
4,5,DNI,86518506,Sandra,González,Morales,1980-11-20,45,Femenino,Viudo,...,850.00,405.03,37044.25,392,Regular,Digital,162,2013-01-08,4,Activo


In [78]:
df_clientes_tra.to_parquet(
    obtener_ruta_archivo("archivos_semi_limpios","semi_limpio_clientes.parquet"),
    index= False)

# Exportando los Registros de Clientes a Revisar

In [79]:
# Union de los registros incorrectos
df_registros_clientes_incorrectos = pd.concat([df_revisar_digitos_documento,df_revisar_fechas_futuras,df_revisar_fechas_registro,df_revisar_fecha_registro_fecha_sucursal])
df_registros_clientes_incorrectos.head()

,cliente_id,tipo_documento,numero_documento,ERROR,fecha_nacimiento,edad,fecha_registro,edad_registro,sucursal_id,fecha_apertura
8,9,DNI,8427109,CANTIDAD DE DIGITOS INCORRECTOS PARA EL TIPO D...,NaT,NaN,NaT,NaN,NaN,NaT
17,18,DNI,2895171,CANTIDAD DE DIGITOS INCORRECTOS PARA EL TIPO D...,NaT,NaN,NaT,NaN,NaN,NaT
46,47,DNI,5486874,CANTIDAD DE DIGITOS INCORRECTOS PARA EL TIPO D...,NaT,NaN,NaT,NaN,NaN,NaT
56,57,DNI,1687339,CANTIDAD DE DIGITOS INCORRECTOS PARA EL TIPO D...,NaT,NaN,NaT,NaN,NaN,NaT
64,65,DNI,6724004,CANTIDAD DE DIGITOS INCORRECTOS PARA EL TIPO D...,NaT,NaN,NaT,NaN,NaN,NaT


In [80]:
# Cargando los registros de clientes que son inconsistentes 
df_registros_clientes_incorrectos.to_parquet(
    obtener_ruta_archivo("archivos_para_revision","revision_clientes.parquet"),
    index=False
)